# 01 · Inventario de SPIDER (solo máscaras y archivos de resumen)

**Propósito.** Describir qué hay en el dataset SPIDER antes de medir nada: cuántas series hay por centro, secuencia, sexo, edad disponible y grosor de corte, y si las máscaras coinciden con lo que declara `overview.csv`.

**Lo que este cuaderno NO hace.** No descarga las imágenes (`images.zip`, 3,7 GB), no calcula ángulos de Cobb y no saca ninguna conclusión clínica. Es un inventario descriptivo para uso exclusivo en investigación.

**Fuente de los datos.** van der Graaf JW *et al.* Lumbar spine segmentation in MR images: a dataset and a public benchmark. *Sci Data* 2024;11:264. doi:[10.1038/s41597-024-03090-w](https://doi.org/10.1038/s41597-024-03090-w). Datos en Zenodo, doi:[10.5281/zenodo.10159290](https://doi.org/10.5281/zenodo.10159290), licencia CC-BY 4.0.

**Datos en Drive.** Los archivos están en `Mi unidad/PRL/content/datos/spider` (con `masks.zip` dentro de la subcarpeta `masks/`). Con `MODO_PRUEBA = True` no se consulta Zenodo: se usan los archivos de Drive y su integridad se comprueba contra las sumas MD5 de la descarga verificada del 2-oct-2026. Con `MODO_PRUEBA = False` se descarga y verifica contra la API de Zenodo.

In [1]:
# 1. Dependencias (Colab ya trae pandas, numpy y requests; falta SimpleITK para leer .mha)
!pip -q install SimpleITK

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 13.8 MB/s eta 0:00:00


In [3]:
# @title
# 2. Configuración
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

ZENODO_RECORD = "10159290"          # versión v4 del registro de SPIDER revisada el 1-oct-2026
ARCHIVOS = ["overview.csv", "radiological_gradings.csv", "masks.zip"]   # images.zip se omite a propósito
DIR_PRL = Path("/content/drive/MyDrive/PRL")
DIR_DATOS = DIR_PRL / "content" / "datos" / "spider"
DIR_SALIDAS = DIR_PRL / "salidas"
MODO_PRUEBA = True                  # True: usa los archivos ya guardados en Drive, sin descargar de Zenodo

# Sumas MD5 de la descarga verificada contra Zenodo
MD5_VERIFICADAS = {
    "overview.csv": "f9a808d2cf67844baa32114a1a769fde",
    "radiological_gradings.csv": "a5b7a6ad7fb4daa0bfa650b716f026f6",
    "masks.zip": "49861ff25a0108b3b69d0ce808ae51dc",
}

def ruta_de(nombre):
    """Busca un archivo en DIR_DATOS y, si no está ahí, en la subcarpeta masks/."""
    for candidata in (DIR_DATOS / nombre, DIR_DATOS / "masks" / nombre):
        if candidata.exists():
            return candidata
    return DIR_DATOS / nombre

assert DIR_DATOS.exists(), f"No existe {DIR_DATOS}; revisa con !ls /content/drive/MyDrive/PRL"
print("DIR_DATOS =", DIR_DATOS)
print("Contenido:", sorted(p.name for p in DIR_DATOS.iterdir()))
DIR_SALIDAS.mkdir(parents=True, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DIR_DATOS = /content/drive/MyDrive/PRL/content/datos/spider
Contenido: ['images', 'masks', 'overview.csv', 'radiological_gradings.csv']


## Descarga con verificación

Primero se consulta la API de Zenodo para registrar la versión del dataset y la suma MD5 de cada archivo; después se descarga solo lo necesario y se comprueba la suma. Si la API no responde, se descarga igual por el enlace directo y se deja constancia de que no hubo verificación. En modo prueba no se descarga nada: se calcula la MD5 de los archivos que ya están en Drive y se compara con las sumas verificadas el 2-oct-2026.

In [4]:
# @title
# 3. Metadatos del registro y descarga verificada
import hashlib, json, requests, datetime

def md5_de(ruta, bloque=1 << 20):
    h = hashlib.md5()
    with open(ruta, "rb") as f:
        for trozo in iter(lambda: f.read(bloque), b""):
            h.update(trozo)
    return h.hexdigest()

registro = {"record": ZENODO_RECORD, "consultado": datetime.datetime.now().isoformat(timespec="seconds"),
            "modo_prueba": MODO_PRUEBA, "dir_datos": str(DIR_DATOS)}
esperado = {}
if MODO_PRUEBA:
    esperado = {k: {"md5": v} for k, v in MD5_VERIFICADAS.items()}
    registro["referencia_md5"] = "descarga verificada contra Zenodo el 2026-10-02"
else:
    try:
        r = requests.get(f"https://zenodo.org/api/records/{ZENODO_RECORD}", timeout=60)
        r.raise_for_status()
        meta = r.json()
        registro.update({
            "titulo": meta.get("metadata", {}).get("title"),
            "version": meta.get("metadata", {}).get("version"),
            "doi": meta.get("doi"),
            "licencia": (meta.get("metadata", {}).get("license") or {}).get("id")
                        or str(meta.get("metadata", {}).get("rights")),
        })
        archivos_api = meta.get("files", [])
        if isinstance(archivos_api, dict):          # por si la API devuelve {"entries": ...}
            archivos_api = list(archivos_api.get("entries", {}).values())
        for a in archivos_api:
            esperado[a["key"]] = {"tamano": a.get("size"), "md5": str(a.get("checksum", "")).replace("md5:", "")}
    except Exception as e:
        print("Aviso: no se pudo leer la API de Zenodo, se descargará sin verificar MD5 ->", e)

    for nombre in ARCHIVOS:
        destino = ruta_de(nombre)
        if destino.exists() and nombre in esperado and md5_de(destino) == esperado[nombre]["md5"]:
            print(f"{nombre}: ya estaba descargado y verificado")
            continue
        url = f"https://zenodo.org/records/{ZENODO_RECORD}/files/{nombre}?download=1"
        with requests.get(url, stream=True, timeout=600) as resp:
            resp.raise_for_status()
            with open(destino, "wb") as f:
                for trozo in resp.iter_content(1 << 20):
                    f.write(trozo)
        print(f"{nombre}: descargado ({destino.stat().st_size/1e6:.1f} MB)")

verificacion = []
for nombre in ARCHIVOS:
    ruta = ruta_de(nombre)
    if not ruta.exists():
        print(f"{nombre}: no se encontró ni en DIR_DATOS ni en DIR_DATOS/masks; se omite su verificación")
        verificacion.append({"archivo": nombre, "ruta": None, "bytes": None, "md5": None, "md5_coincide": None})
        continue
    suma = md5_de(ruta)
    ok = (suma == esperado[nombre]["md5"]) if nombre in esperado else None
    verificacion.append({"archivo": nombre, "ruta": str(ruta.relative_to(DIR_DATOS)), "bytes": ruta.stat().st_size,
                         "md5": suma, "md5_coincide": ok})
    assert ok is not False, f"La suma MD5 de {nombre} no coincide con la referencia: archivo corrupto o distinto"
registro["archivos"] = verificacion
print(json.dumps(registro, indent=2, ensure_ascii=False))

{
  "record": "10159290",
  "consultado": "2026-10-03T22:15:24",
  "modo_prueba": true,
  "dir_datos": "/content/drive/MyDrive/PRL/content/datos/spider",
  "referencia_md5": "descarga verificada contra Zenodo el 2026-10-02",
  "archivos": [
    {
      "archivo": "overview.csv",
      "ruta": "overview.csv",
      "bytes": 119856,
      "md5": "f9a808d2cf67844baa32114a1a769fde",
      "md5_coincide": true
    },
    {
      "archivo": "radiological_gradings.csv",
      "ruta": "radiological_gradings.csv",
      "bytes": 34417,
      "md5": "a5b7a6ad7fb4daa0bfa650b716f026f6",
      "md5_coincide": true
    },
    {
      "archivo": "masks.zip",
      "ruta": "masks/masks.zip",
      "bytes": 58222897,
      "md5": "49861ff25a0108b3b69d0ce808ae51dc",
      "md5_coincide": true
    }
  ]
}


In [5]:
# @title
# 4. Descomprimir solo las máscaras (la primera vez escribe 447 archivos en Drive y tarda unos minutos)
import zipfile
dir_mascaras = DIR_DATOS / "masks"
if not any(dir_mascaras.rglob("*.mha")):
    with zipfile.ZipFile(ruta_de("masks.zip")) as z:
        z.extractall(dir_mascaras)
mascaras = sorted(dir_mascaras.rglob("*.mha"))
print(len(mascaras), "máscaras .mha encontradas; ejemplos:", [m.name for m in mascaras[:4]])

447 máscaras .mha encontradas; ejemplos: ['100_t1.mha', '100_t2.mha', '101_t1.mha', '101_t2.mha']


## Tabla de resumen (`overview.csv`)

El archivo trae una fila por serie. El identificador `new_file_name` combina el estudio y la secuencia (por ejemplo `1_t1`, `1_t2`); según el artículo, todas las series de un mismo estudio comparten identificador y cada estudio corresponde a un paciente distinto.

In [6]:
# @title
# 5. Leer y limpiar overview.csv
import pandas as pd, numpy as np, re

ov = pd.read_csv(DIR_DATOS / "overview.csv", dtype={"birth_date": str, "new_file_name": str})
print("Filas:", len(ov), "| Columnas:", len(ov.columns))

ov["serie"] = ov["new_file_name"].astype(str).str.strip()
ov["estudio"] = ov["serie"].str.split("_", n=1).str[0]
sufijo = ov["serie"].str.split("_", n=1).str[1].str.lower()
ov["secuencia"] = np.where(sufijo.str.contains("space", na=False), "t2_SPACE", sufijo)
ov["sexo"] = ov["sex"].astype(str).str.strip().str.upper().replace({"NAN": np.nan, "": np.nan})

print("\nSecuencias encontradas:")
print(ov["secuencia"].value_counts(dropna=False).to_string())

Filas: 447 | Columnas: 39

Secuencias encontradas:
secuencia
t2          210
t1          196
t2_SPACE     41


## Centro: SPIDER no publica el hospital de origen

`overview.csv` no tiene columna de centro. Por eso aquí se construye un **proxy**, y se debe reportar como tal:

1. Según el artículo, todos los estudios del centro universitario (UMC) incluyen una serie T2 SPACE y los otros tres hospitales solo aportan T1 o T2 estándar. Un estudio con serie SPACE se marca como `UMC (inferido por SPACE)`. Esto es una **inferencia** a partir del texto del artículo, no un dato del dataset.
2. El resto se agrupa por escáner (fabricante + modelo + campo). Un hospital puede tener varios escáneres y un modelo puede estar en varios hospitales, así que el grupo de escáner **no es** el centro.

**Comprobación manual pendiente:** comparar los conteos de abajo con la tabla por hospital del artículo de *Sci Data* y anotar si coinciden.

In [7]:
# @title
# 6. Proxy de centro
estudios_con_space = set(ov.loc[ov["secuencia"] == "t2_SPACE", "estudio"])
ov["escaner"] = (ov["Manufacturer"].astype(str).str.strip() + " | " +
                 ov["ManufacturerModelName"].astype(str).str.strip() + " | " +
                 ov["MagneticFieldStrength"].astype(str) + " T")

def proxy_centro(fila):
    if fila["estudio"] in estudios_con_space:
        return "UMC (inferido por SPACE)"
    return "otro: " + fila["escaner"]
ov["centro_proxy"] = ov.apply(proxy_centro, axis=1)

print("Estudios con SPACE:", len(estudios_con_space))
print(pd.crosstab(ov["centro_proxy"], ov["secuencia"], margins=True, margins_name="Total").to_string())

Estudios con SPACE: 41
secuencia                                         t1   t2  t2_SPACE  Total
centro_proxy                                                              
UMC (inferido por SPACE)                          39   39        41    119
otro: Philips Healthcare | Ingenia | 3.0 T        90   90         0    180
otro: Philips Medical Systems | Ingenia | 1.5 T   12   18         0     30
otro: Philips Medical Systems | Ingenia | 3.0 T    0    1         0      1
otro: SIEMENS | Aera | 1.5 T                      17   16         0     33
otro: SIEMENS | Avanto | 1.5 T                     1    1         0      2
otro: SIEMENS | Avanto_fit | 1.5 T                11   24         0     35
otro: SIEMENS | Skyra_fit | 3.0 T                  5    5         0     10
otro: SIEMENS | SymphonyTim | 1.5 T               21   16         0     37
Total                                            196  210        41    447


## Edad: qué hay realmente

El artículo dice que la edad se da "para los pacientes en que estaba disponible". La columna se llama `birth_date`.

In [10]:
# @title
# 7. Disponibilidad de edad / fecha de nacimiento (a nivel de paciente)
pac = ov.groupby("estudio").agg(sexo=("sexo", "first"),
                                birth_date=("birth_date", "first"),
                                subset=("subset", lambda s: "|".join(sorted(set(s.astype(str))))),
                                n_series=("serie", "count"),
                                centro_proxy=("centro_proxy", "first"))
tiene_edad = pac["birth_date"].notna() & (pac["birth_date"].astype(str).str.strip() != "")
print(f"Pacientes: {len(pac)} | con birth_date: {tiene_edad.sum()} | sin dato: {(~tiene_edad).sum()}")

def formato(v):
    v = str(v).strip()
    if re.fullmatch(r"\d{4}", v): return "año (AAAA)"
    if re.fullmatch(r"\d{3}Y", v): return "edad DICOM (nnnY)"
    if re.fullmatch(r"\d{8}", v): return "fecha DICOM (AAAAMMDD)"
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}.*", v): return "fecha ISO"
    if re.fullmatch(r"\d+(\.\d+)?", v): return "número (¿edad?)"
    return "otro"
if tiene_edad.any():
    print("\nFormatos encontrados:")
    print(pac.loc[tiene_edad, "birth_date"].map(formato).value_counts().to_string())
    edades = pd.to_numeric(pac.loc[tiene_edad, "birth_date"], errors="coerce")
    print(f"\nEdades encontradas (birth_date interpretado como edad en años; n = {edades.notna().sum()}):")
    print(edades.describe().round(1).to_string())
    print("Primeros valores:", edades.head(5).tolist())

Pacientes: 218 | con birth_date: 41 | sin dato: 177

Formatos encontrados:
birth_date
número (¿edad?)    41

Edades encontradas (birth_date interpretado como edad en años; n = 41):
count    41.0
mean     58.4
std      16.7
min      14.0
25%      49.0
50%      59.0
75%      72.0
max      84.0
Primeros valores: [59.0, 77.0, 49.0, 44.0, 63.0]


In [22]:
# @title
# 8. Control de partición a nivel de paciente
mezclados = pac[pac["subset"].str.contains(r"\|")]
print("Pacientes con series en más de un subconjunto:", len(mezclados))
print(pac["subset"].value_counts().to_string())

Pacientes con series en más de un subconjunto: 0
subset
training      179
validation     39


## Máscaras: geometría y etiquetas

Se lee la cabecera de cada `.mha` (tamaño, espaciado, orientación) y sus etiquetas. El eje izquierda-derecha es el que importa para un Cobb coronal reconstruido desde cortes sagitales, porque ahí cae el grosor de corte.

Convención de etiquetas observada en el script público de DIAG (`compute-Cobb-angle.py`): vértebras < 100, canal = 100, discos > 200, numeradas de abajo hacia arriba y **sin** equivalencia anatómica garantizada (la etiqueta 1 no es necesariamente L5). El cuaderno la comprueba en vez de asumirla.

In [11]:
# @title
# 9. Recorrer las máscaras
import SimpleITK as sitk

filas = []
for ruta in mascaras:
    lector = sitk.ImageFileReader()
    lector.SetFileName(str(ruta))
    lector.ReadImageInformation()
    esp = lector.GetSpacing()
    D = np.array(lector.GetDirection()).reshape(3, 3)          # columnas = direcciones de los ejes del volumen (LPS)
    eje_lr = int(np.argmax(np.abs(D[0, :])))                    # eje cuya dirección es más paralela a izquierda-derecha
    arr = sitk.GetArrayFromImage(sitk.ReadImage(str(ruta)))
    etiquetas = np.unique(arr)
    etiquetas = etiquetas[etiquetas > 0]
    filas.append({
        "serie": ruta.stem,
        "tamano_vox": tuple(int(x) for x in lector.GetSize()),
        "espaciado_mm": tuple(round(float(x), 3) for x in esp),
        "eje_lr": eje_lr,
        "espaciado_lr_mm": round(float(esp[eje_lr]), 3),
        "n_vertebras_mascara": int(((etiquetas > 0) & (etiquetas < 100)).sum()),
        "tiene_canal": bool((etiquetas == 100).any()),
        "n_discos_mascara": int((etiquetas > 200).sum()),
        "etiquetas_otras": [int(e) for e in etiquetas if 100 < e <= 200],
    })
mk = pd.DataFrame(filas)
print(len(mk), "máscaras leídas")
print("Etiquetas fuera de la convención esperada:", int(mk["etiquetas_otras"].map(len).gt(0).sum()), "series")

447 máscaras leídas
Etiquetas fuera de la convención esperada: 0 series


In [12]:
# @title
# 10. Unir con overview.csv y comprobar consistencia
inv = ov.merge(mk, on="serie", how="outer", indicator=True)
print("Series solo en overview.csv:", (inv["_merge"] == "left_only").sum())
print("Máscaras sin fila en overview.csv:", (inv["_merge"] == "right_only").sum())

inv["dif_vertebras"] = inv["n_vertebras_mascara"] - inv["num_vertebrae"]
inv["dif_discos"] = inv["n_discos_mascara"] - inv["num_discs"]
print("Series donde el conteo de vértebras no coincide:", int((inv["dif_vertebras"].fillna(0) != 0).sum()))
print("Series donde el conteo de discos no coincide:", int((inv["dif_discos"].fillna(0) != 0).sum()))

# Grosor de corte: el declarado en DICOM y el espaciado real en el eje izquierda-derecha de la máscara
inv["SliceThickness"] = pd.to_numeric(inv["SliceThickness"], errors="coerce")
inv["SpacingBetweenSlices"] = pd.to_numeric(inv["SpacingBetweenSlices"], errors="coerce")
cortes = [0, 1, 3, 4, 5, 100]
etiquetas_corte = ["<1 mm", "1–3 mm", "3–4 mm", "4–5 mm", "≥5 mm"]
inv["grosor_lr_rango"] = pd.cut(inv["espaciado_lr_mm"], bins=cortes, labels=etiquetas_corte, right=False)

Series solo en overview.csv: 0
Máscaras sin fila en overview.csv: 0
Series donde el conteo de vértebras no coincide: 0
Series donde el conteo de discos no coincide: 0


## Tablas del inventario

In [19]:
# @title
# 11. Tablas finales
def mostrar(titulo, tabla):
    print(f"\n=== {titulo} ===")
    print(tabla.to_string())

base = inv[inv["_merge"] != "right_only"]

t_centro = pd.crosstab(base["centro_proxy"], base["secuencia"], margins=True, margins_name="Total")
t_sexo = pd.crosstab(pac["centro_proxy"], pac["sexo"].fillna("sin dato"), margins=True, margins_name="Total")
t_edad = pd.crosstab(pac["centro_proxy"], np.where(tiene_edad, "con fecha/edad", "sin dato"), margins=True, margins_name="Total")
t_grosor = pd.crosstab(base["grosor_lr_rango"], base["secuencia"], margins=True, margins_name="Total")
t_grosor_desc = base.groupby("secuencia")[["SliceThickness", "SpacingBetweenSlices", "espaciado_lr_mm"]].describe().round(2)

mostrar("Series por centro (proxy) y secuencia", t_centro)
mostrar("Pacientes por centro (proxy) y sexo", t_sexo)
mostrar("Pacientes con dato de edad", t_edad)
mostrar("Series por espaciado entre cortes en el eje izquierda-derecha (máscara)", t_grosor)
mostrar("Grosor declarado vs espaciado real (mm)", t_grosor_desc)

resumen = pd.DataFrame([{
    "pacientes": len(pac),
    "series": len(base),
    "series_T1": int((base["secuencia"] == "t1").sum()),
    "series_T2": int((base["secuencia"] == "t2").sum()),
    "series_T2_SPACE": int((base["secuencia"] == "t2_SPACE").sum()),
    "pacientes_mujeres": int((pac["sexo"] == "F").sum()),
    "pacientes_con_edad": int(tiene_edad.sum()),
    "vertebras_en_mascaras": int(base["n_vertebras_mascara"].sum()),
    "discos_en_mascaras": int(base["n_discos_mascara"].sum()),
}])
mostrar("Resumen global", resumen.T)
print("\nReferencia del artículo (Sci Data 2024): 218 pacientes, 447 series, 63 % mujeres, 3125 vértebras, 3147 discos.")


=== Series por centro (proxy) y secuencia ===
secuencia                                         t1   t2  t2_SPACE  Total
centro_proxy                                                              
UMC (inferido por SPACE)                          39   39        41    119
otro: Philips Healthcare | Ingenia | 3.0 T        90   90         0    180
otro: Philips Medical Systems | Ingenia | 1.5 T   12   18         0     30
otro: Philips Medical Systems | Ingenia | 3.0 T    0    1         0      1
otro: SIEMENS | Aera | 1.5 T                      17   16         0     33
otro: SIEMENS | Avanto | 1.5 T                     1    1         0      2
otro: SIEMENS | Avanto_fit | 1.5 T                11   24         0     35
otro: SIEMENS | Skyra_fit | 3.0 T                  5    5         0     10
otro: SIEMENS | SymphonyTim | 1.5 T               21   16         0     37
Total                                            196  210        41    447

=== Pacientes por centro (proxy) y sexo ===
sexo    

In [18]:
print(base.columns.tolist())
print(pac.columns.tolist())

['new_file_name', 'num_vertebrae', 'num_discs', 'sex', 'birth_date', 'subset', 'AngioFlag', 'BodyPartExamined', 'DeviceSerialNumber', 'EchoNumbers', 'EchoTime', 'EchoTrainLength', 'FlipAngle', 'ImagedNucleus', 'ImagingFrequency', 'InPlanePhaseEncodingDirection', 'MRAcquisitionType', 'MagneticFieldStrength', 'Manufacturer', 'ManufacturerModelName', 'NumberOfPhaseEncodingSteps', 'PercentPhaseFieldOfView', 'PercentSampling', 'PhotometricInterpretation', 'PixelBandwidth', 'PixelSpacing', 'RepetitionTime', 'SAR', 'SamplesPerPixel', 'ScanningSequence', 'SequenceName', 'SeriesDescription', 'SliceThickness', 'SoftwareVersions', 'SpacingBetweenSlices', 'SpecificCharacterSet', 'TransmitCoilName', 'WindowCenter', 'WindowWidth', 'serie', 'estudio', 'secuencia', 'sexo', 'escaner', 'centro_proxy', 'tamano_vox', 'espaciado_mm', 'eje_lr', 'espaciado_lr_mm', 'n_vertebras_mascara', 'tiene_canal', 'n_discos_mascara', 'etiquetas_otras', '_merge', 'dif_vertebras', 'dif_discos', 'grosor_lr_rango']
['sexo', 

In [20]:
# @title
# 12. Guardar salidas y entorno
import platform, sys
cols = ["serie", "estudio", "secuencia", "subset", "sexo", "centro_proxy", "escaner",
        "SliceThickness", "SpacingBetweenSlices", "espaciado_lr_mm", "grosor_lr_rango",
        "tamano_vox", "espaciado_mm", "num_vertebrae", "n_vertebras_mascara",
        "num_discs", "n_discos_mascara", "tiene_canal", "_merge"]
inv[cols].to_csv(DIR_SALIDAS / "inventario_series.csv", index=False)
pac.to_csv(DIR_SALIDAS / "inventario_pacientes.csv")
with pd.ExcelWriter(DIR_SALIDAS / "inventario_tablas.xlsx") as xw:
    resumen.T.to_excel(xw, sheet_name="resumen")
    t_centro.to_excel(xw, sheet_name="centro_x_secuencia")
    t_sexo.to_excel(xw, sheet_name="centro_x_sexo")
    t_edad.to_excel(xw, sheet_name="edad_disponible")
    t_grosor.to_excel(xw, sheet_name="grosor_x_secuencia")
entorno = {**registro, "python": sys.version.split()[0], "plataforma": platform.platform(),
           "pandas": pd.__version__, "numpy": np.__version__, "SimpleITK": sitk.Version_VersionString()}
(DIR_SALIDAS / "entorno.json").write_text(json.dumps(entorno, indent=2, ensure_ascii=False, default=str))
print("Archivos en salidas/:", sorted(p.name for p in DIR_SALIDAS.iterdir()))

Archivos en salidas/: ['entorno.json', 'inventario_pacientes.csv', 'inventario_series.csv', 'inventario_tablas.xlsx']


In [21]:
# 13. descargar las salidas como .zip
try:
    from google.colab import files
    import shutil
    shutil.make_archive("/content/salidas_inventario", "zip", DIR_SALIDAS)
    files.download("/content/salidas_inventario.zip")
except ImportError:
    print("No estás en Colab; las salidas quedaron en", DIR_SALIDAS)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>